# Q2.2 — Cross-Variety Evaluation

Train DistilBERT on one English variety, test on all three.
Produces a 3x3 Macro-F1 matrix for **Sarcasm** detection.

**Key question:** Does training on inner-circle varieties (en-UK, en-AU) transfer better than outer-circle (en-IN)?

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 42
VARIETIES = ["en-UK", "en-AU", "en-IN"]
MODEL_NAME = "distilbert-base-uncased"

np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Varieties: {VARIETIES}")

Device: mps
Model: distilbert-base-uncased
Varieties: ['en-UK', 'en-AU', 'en-IN']


## 1 — Load & Split Data by Variety

In [2]:
dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_df = dataset["train"].to_pandas()
val_df   = dataset["validation"].to_pandas()
test_df  = dataset["test"].to_pandas()

for df in [train_df, val_df, test_df]:
    df["Sentiment"] = df["Sentiment"].astype(int)
    df["Sarcasm"] = df["Sarcasm"].astype(int)

# Split by variety
splits = {}
for v in VARIETIES:
    splits[v] = {
        "train": train_df[train_df["variety"] == v].reset_index(drop=True),
        "val":   val_df[val_df["variety"] == v].reset_index(drop=True),
        "test":  test_df[test_df["variety"] == v].reset_index(drop=True),
    }

for v in VARIETIES:
    tr = splits[v]["train"]
    va = splits[v]["val"]
    te = splits[v]["test"]
    sarc_rate = tr["Sarcasm"].mean() * 100
    print(f"{v}: train={len(tr)}, val={len(va)}, test={len(te)}, sarc_rate={sarc_rate:.1f}%")

en-UK: train=1203, val=101, test=700, sarc_rate=7.6%
en-AU: train=1145, val=95, test=667, sarc_rate=29.4%
en-IN: train=1399, val=117, test=816, sarc_rate=6.8%


## 2 — Cross-Variety Training Function

For each variety, fine-tune DistilBERT on its training data, then evaluate on **all three** test sets.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

def train_variety_model(train_variety, seed=SEED):
    """Train DistilBERT on one variety's data for Sarcasm detection."""
    print(f"\n{'#'*60}")
    print(f"# Training on {train_variety} (seed={seed})")
    print(f"{'#'*60}")

    tr = splits[train_variety]["train"]
    va = splits[train_variety]["val"]

    train_ds = Dataset.from_pandas(tr[["text", "Sarcasm"]].rename(columns={"Sarcasm": "label"}))
    val_ds   = Dataset.from_pandas(va[["text", "Sarcasm"]].rename(columns={"Sarcasm": "label"}))
    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds   = val_ds.map(tokenize_fn, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    # Class weights
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=tr["Sarcasm"].values)
    class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
            loss = loss_fct(logits, labels)
            return (loss, outputs) if return_outputs else loss

    output_dir = f"./results/cross_variety_{train_variety}_seed{seed}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=seed,
        fp16=False,
        report_to="none",
        logging_steps=50,
        disable_tqdm=True,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[],
    )

    trainer.train()

    # Evaluate on ALL three test varieties
    results = {}
    for test_v in VARIETIES:
        te = splits[test_v]["test"]
        test_ds = Dataset.from_pandas(te[["text", "Sarcasm"]].rename(columns={"Sarcasm": "label"}))
        test_ds = test_ds.map(tokenize_fn, batched=True)

        preds = trainer.predict(test_ds)
        y_pred = np.argmax(preds.predictions, axis=-1)
        y_true = preds.label_ids

        macro_f1 = f1_score(y_true, y_pred, average="macro")
        results[test_v] = {
            "macro_f1": macro_f1,
            "y_pred": y_pred,
            "y_true": y_true,
        }
        print(f"  Test on {test_v}: Macro-F1 = {macro_f1:.4f}")

    return results

print("Training function defined.")

Training function defined.


## 3 — Run Cross-Variety Experiments (Seed 42)

In [4]:
# Train on each variety, test on all — seed 42
cross_results_42 = {}
for train_v in VARIETIES:
    cross_results_42[train_v] = train_variety_model(train_v, seed=42)


############################################################
# Training on en-UK (seed=42)
############################################################


Map:   0%|          | 0/1203 [00:00<?, ? examples/s]

Map: 100%|██████████| 1203/1203 [00:00<00:00, 20264.45 examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Map: 100%|██████████| 101/101 [00:00<00:00, 14422.25 examples/s]

...

Map: 100%|██████████| 667/667 [00:00<00:00, 18912.80 examples/s]

  Test on en-AU: Macro-F1 = 0.4365


Map:   0%|          | 0/816 [00:00<?, ? examples/s]

Map: 100%|██████████| 816/816 [00:00<00:00, 30075.15 examples/s]

  Test on en-IN: Macro-F1 = 0.5963


## 4 — Run Cross-Variety Experiments (Seed 123)

In [5]:
# Train on each variety, test on all — seed 123
cross_results_123 = {}
for train_v in VARIETIES:
    cross_results_123[train_v] = train_variety_model(train_v, seed=123)


############################################################
# Training on en-UK (seed=123)
############################################################


Map:   0%|          | 0/1203 [00:00<?, ? examples/s]

Map: 100%|██████████| 1203/1203 [00:00<00:00, 19463.24 examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Map: 100%|██████████| 101/101 [00:00<00:00, 13197.44 examples/s]

...

Map: 100%|██████████| 667/667 [00:00<00:00, 21485.63 examples/s]

  Test on en-AU: Macro-F1 = 0.4259


Map:   0%|          | 0/816 [00:00<?, ? examples/s]

Map: 100%|██████████| 816/816 [00:00<00:00, 33218.02 examples/s]

  Test on en-IN: Macro-F1 = 0.6296


## 5 — Cross-Variety Macro-F1 Matrices

In [6]:
def build_f1_matrix(results_dict, seed_label):
    """Build a 3x3 DataFrame of Macro-F1 scores."""
    matrix = pd.DataFrame(index=VARIETIES, columns=VARIETIES, dtype=float)
    for train_v in VARIETIES:
        for test_v in VARIETIES:
            matrix.loc[train_v, test_v] = results_dict[train_v][test_v]["macro_f1"]
    matrix.index.name = "Train on"
    matrix.columns.name = "Test on"
    print(f"\nCross-Variety Macro-F1 Matrix — Seed {seed_label}:")
    print(matrix.to_string(float_format="{:.4f}".format))
    return matrix

matrix_42  = build_f1_matrix(cross_results_42, 42)
matrix_123 = build_f1_matrix(cross_results_123, 123)

# Average matrix
matrix_avg = (matrix_42 + matrix_123) / 2
print(f"\nAverage Cross-Variety Macro-F1 Matrix:")
print(matrix_avg.to_string(float_format="{:.4f}".format))


Cross-Variety Macro-F1 Matrix — Seed 42:
Test on   en-UK  en-AU  en-IN
Train on                     
en-UK    0.6608 0.5759 0.5927
en-AU    0.5892 0.7089 0.4679
en-IN    0.6126 0.4365 0.5963

Cross-Variety Macro-F1 Matrix — Seed 123:
Test on   en-UK  en-AU  en-IN
Train on                     
en-UK    0.6662 0.6406 0.5364
en-AU    0.5944 0.7075 0.4732
en-IN    0.5473 0.4259 0.6296

Average Cross-Variety Macro-F1 Matrix:
Test on   en-UK  en-AU  en-IN
Train on                     
en-UK    0.6635 0.6083 0.5645
en-AU    0.5918 0.7082 0.4706
en-IN    0.5799 0.4312 0.6129


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, matrix, title in zip(
    axes,
    [matrix_42, matrix_123, matrix_avg],
    ["Seed 42", "Seed 123", "Average"],
):
    sns.heatmap(
        matrix.astype(float), annot=True, fmt=".3f",
        cmap="YlGn", ax=ax, vmin=0.4, vmax=0.85,
    )
    ax.set_title(f"Cross-Variety Macro-F1\n({title})")
    ax.set_ylabel("Trained on")
    ax.set_xlabel("Tested on")

plt.suptitle("Sarcasm Detection: Cross-Variety Transfer", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("q22_cross_variety_matrix.png", bbox_inches="tight")
plt.show()

## 6 — Diagonal (Within-Variety) vs Off-Diagonal (Cross-Variety)

In [8]:
print("Within-variety (diagonal) vs Cross-variety (off-diagonal) performance:")
print("="*70)

for v in VARIETIES:
    within = matrix_avg.loc[v, v]
    cross_vals = [matrix_avg.loc[v, other] for other in VARIETIES if other != v]
    avg_cross = np.mean(cross_vals)
    drop = within - avg_cross
    print(f"{v}:  within={within:.4f}  avg_cross={avg_cross:.4f}  drop={drop:.4f} ({drop/within*100:.1f}%)")

# Inner-circle transfer
inner_pairs = [("en-UK", "en-AU"), ("en-AU", "en-UK")]
outer_pairs = [("en-UK", "en-IN"), ("en-AU", "en-IN"), ("en-IN", "en-UK"), ("en-IN", "en-AU")]

inner_f1 = np.mean([matrix_avg.loc[t, e] for t, e in inner_pairs])
outer_f1 = np.mean([matrix_avg.loc[t, e] for t, e in outer_pairs])

print(f"\nInner-circle transfer (en-UK<->en-AU): {inner_f1:.4f}")
print(f"Inner<->Outer transfer:                {outer_f1:.4f}")
print(f"Gap:                                   {inner_f1 - outer_f1:.4f}")

# Same-variety average
diag_avg = np.mean([matrix_avg.loc[v, v] for v in VARIETIES])
print(f"\nAverage same-variety F1: {diag_avg:.4f}")
print(f"Average cross-variety F1: {outer_f1:.4f}")

Within-variety (diagonal) vs Cross-variety (off-diagonal) performance:
en-UK:  within=0.6635  avg_cross=0.5864  drop=0.0771 (11.6%)
en-AU:  within=0.7082  avg_cross=0.5312  drop=0.1770 (25.0%)
en-IN:  within=0.6129  avg_cross=0.5056  drop=0.1074 (17.5%)

Inner-circle transfer (en-UK<->en-AU): 0.6000
Inner<->Outer transfer:                0.5116
Gap:                                   0.0885

Average same-variety F1: 0.6616
Average cross-variety F1: 0.5116


## 7 — Per-Class Analysis by Variety

In [9]:
# Per-class F1 for within-variety models
print("Per-class F1 for within-variety models (Sarcasm task):")
print("="*60)

for v in VARIETIES:
    y_true = cross_results_42[v][v]["y_true"]
    y_pred = cross_results_42[v][v]["y_pred"]
    report = classification_report(y_true, y_pred, target_names=["Not Sarcastic", "Sarcastic"], digits=4)
    print(f"\n--- Trained & tested on {v} ---")
    print(report)

Per-class F1 for within-variety models (Sarcasm task):

--- Trained & tested on en-UK ---
               precision    recall  f1-score   support

Not Sarcastic     0.9551    0.9196    0.9370       647
    Sarcastic     0.3247    0.4717    0.3846        53

     accuracy                         0.8857       700
    macro avg     0.6399    0.6957    0.6608       700
 weighted avg     0.9073    0.8857    0.8952       700


--- Trained & tested on en-AU ---
               precision    recall  f1-score   support

Not Sarcastic     0.8981    0.6921    0.7818       471
    Sarcastic     0.5230    0.8112    0.6360       196

     accuracy                         0.7271       667
    macro avg     0.7105    0.7517    0.7089       667
 weighted avg     0.7879    0.7271    0.7389       667


--- Trained & tested on en-IN ---
               precision    recall  f1-score   support

Not Sarcastic     0.9482    0.9158    0.9317       760
    Sarcastic     0.2195    0.3214    0.2609        56

     ac

## 8 — Stability Check: Seed 42 vs Seed 123

In [10]:
print("Stability: Absolute difference between seed 42 and seed 123")
print("="*60)

diff_matrix = (matrix_42 - matrix_123).abs()
print(diff_matrix.to_string(float_format="{:.4f}".format))
print(f"\nMean absolute diff: {diff_matrix.values.mean():.4f}")
print(f"Max absolute diff:  {diff_matrix.values.max():.4f}")

Stability: Absolute difference between seed 42 and seed 123
Test on   en-UK  en-AU  en-IN
Train on                     
en-UK    0.0054 0.0647 0.0564
en-AU    0.0052 0.0014 0.0053
en-IN    0.0653 0.0106 0.0333

Mean absolute diff: 0.0275
Max absolute diff:  0.0653


## Summary

| Finding | Detail |
|---|---|
| Task | Sarcasm detection (binary) |
| Model | DistilBERT (same as Q2.1 best) |
| Training | Per-variety fine-tuning with class-weighted loss |
| Seeds | 42 and 123 for stability |
| Metric | Macro-F1 |
| Key question | Does inner-circle transfer beat inner<->outer? |

Saved figure: `q22_cross_variety_matrix.png`